In [1]:
from datasets import load_dataset
import numpy as np

/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# https://huggingface.co/datasets/amazon_polarity
# takes a long time to process, you may want to try it yourself
# dataset = load_dataset("amazon_polarity")

In [3]:
raw_datasets = load_dataset("glue", "sst2")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [4]:
raw_datasets['train']

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 67349
})

In [5]:
dir(raw_datasets['train'])

['_TF_DATASET_REFS',
 '__class__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getitems__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_local_temp_path',
 '_check_index_is_initialized',
 '_data',
 '_estimate_nbytes',
 '_fast_select_column',
 '_fingerprint',
 '_format_columns',
 '_format_kwargs',
 '_format_type',
 '_generate_tables_from_cache_file',
 '_generate_tables_from_shards',
 '_get_cache_file_path',
 '_get_output_signature',
 '_getitem',
 '_indexes',
 '_indices',
 '_info',
 '_map_single',
 '_new_dataset_with_indices',
 '_output_all_columns',
 '_push_parquet_shards_to_hub',
 '_push_parquet_sha

In [6]:
type(raw_datasets["train"])

datasets.arrow_dataset.Dataset

In [7]:
raw_datasets['train'].data

MemoryMappedTable
sentence: string
label: int64
idx: int32
----
sentence: [["hide new secretions from the parental units ","contains no wit , only labored gags ","that loves its characters and communicates something rather beautiful about human nature ","remains utterly satisfied to remain the same throughout ","on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ",...,"you wish you were at home watching that movie instead of in the theater watching this one ","'s no point in extracting the bare bones of byatt 's plot for purposes of bland hollywood romance ","underdeveloped ","the jokes are flat ","a heartening tale of small victories "],["suspense , intriguing characters and bizarre bank robberies , ","a gritty police thriller with all the dysfunctional family dynamics one could wish for ","with a wonderful ensemble cast of characters that bring the routine day to day struggles of the wo (... 20 chars omitted)","nonetheless appreciates the art and reveals a music

In [8]:
raw_datasets['train'][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

In [9]:
raw_datasets['train'][50000:50003]

{'sentence': ['glow ',
  'a classical dramatic animated feature ',
  'best espionage picture '],
 'label': [1, 1, 1],
 'idx': [50000, 50001, 50002]}

In [10]:
raw_datasets['train'].features

{'sentence': Value('string'),
 'label': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32')}

In [11]:
from transformers import AutoTokenizer

In [12]:
# checkpoint = "bert-base-uncased"
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [13]:
tokenized_sentences = tokenizer(raw_datasets["train"][0:3]["sentence"])
from pprint import pprint
pprint(tokenized_sentences)

{'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
 'input_ids': [[101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102],
               [101,
                3397,
                2053,
                15966,
                1010,
                2069,
                4450,
                2098,
                18201,
                2015,
                102],
               [101,
                2008,
                7459,
                2049,
                3494,
                1998,
                10639,
                2015,
                2242,
                2738,
                3376,
                2055,
                2529,
                3267,
                102]]}


In [14]:
def tokenize_fn(batch):
    return tokenizer(batch['sentence'], truncation=True)

In [15]:
tokenized_datasets = raw_datasets.map(
    tokenize_fn,
    batched=True,
)

In [16]:
from transformers import TrainingArguments

In [17]:
training_args = TrainingArguments(
    'my_trainer',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=1,
)

In [18]:
from transformers import AutoModelForSequenceClassification

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
type(model)

transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification

In [21]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [22]:
from torchinfo import summary

# summary(model, input_size = (16,512), dtypes =['torch.IntTensor], device='cpu')
summary(model)


Layer (type:depth-idx)                                  Param #
DistilBertForSequenceClassification                     --
├─DistilBertModel: 1-1                                  --
│    └─Embeddings: 2-1                                  --
│    │    └─Embedding: 3-1                              23,440,896
│    │    └─Embedding: 3-2                              393,216
│    │    └─LayerNorm: 3-3                              1,536
│    │    └─Dropout: 3-4                                --
│    └─Transformer: 2-2                                 --
│    │    └─ModuleList: 3-5                             42,527,232
├─Linear: 1-2                                           590,592
├─Linear: 1-3                                           1,538
├─Dropout: 1-4                                          --
Total params: 66,955,010
Trainable params: 66,955,010
Non-trainable params: 0

In [23]:
params_before = []
for name, p in model.named_parameters():
    params_before.append(p.detach().cpu().numpy())

In [24]:
from transformers import Trainer

In [25]:
import evaluate

In [26]:
metric = evaluate.load('glue', 'sst2')

In [27]:
metric.compute(predictions=[1,0,1], references=[1,0,0])

{'accuracy': 0.6666666666666666}

In [28]:
def compute_metrics(logits_and_labels):
    # metric = load_metric("glue", "sst2")
    logits, labels = logits_and_labels
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/var/folders/8t/4ttt_tc53hs994xcrmjg2p880000gn/T/ipykernel_63552/3893836303.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [30]:
trainer.train()

/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.200200,0.340762,0.909404


TrainOutput(global_step=8419, training_loss=0.26123963054816024, metrics={'train_runtime': 1635.7139, 'train_samples_per_second': 41.174, 'train_steps_per_second': 5.147, 'total_flos': 517212489917652.0, 'train_loss': 0.26123963054816024, 'epoch': 1.0})

In [31]:
trainer.save_model("my_saved_model")

In [32]:
!ls

1.Pipeline_Sentiment_Analysis.ipynb
10.Fine-Tuning-Sentiment-Analysis.ipynb
2.Pipeline-Text-Generation.ipynb
3.Pipeline-Masked-Language-Modeling.ipynb
4.Pipeline_NER.ipynb
5.Pipeline-Summarization.ipynb
6.Pipeline-Translation.ipynb
7.Pipeline-Question-Answering.ipynb
8.Pipeline-Zero-Shot-Classification.ipynb
9.Models_and_Tokenizers.ipynb
Note.md
Resources
data
images
machine_learning_examples
my_saved_model
my_trainer


In [33]:
!ls my_saved_model

config.json             tokenizer.json          vocab.txt
model.safetensors       tokenizer_config.json
special_tokens_map.json training_args.bin


In [34]:
from transformers import pipeline

In [35]:
newmodel = pipeline("text-classification", model='my_saved_model', device='mps')

Device set to use mps


In [36]:
newmodel('This movie is great!')

[{'label': 'LABEL_1', 'score': 0.9995844960212708}]

In [37]:
newmodel('This movie sucks')

[{'label': 'LABEL_0', 'score': 0.9980302453041077}]

In [38]:
!cat my_saved_model/config.json

{
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "problem_type": "single_label_classification",
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.57.6",
  "vocab_size": 30522
}


In [39]:
import json

In [40]:
config_path = 'my_saved_model/config.json'
with open(config_path, 'r') as f:
    j = json.load(f)

j['id2label'] = {0: 'negative', 1: 'positive'}

with open(config_path, 'w') as f:
    json.dump(j, f,indent=2)

In [41]:
!cat my_saved_model/config.json

{
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "problem_type": "single_label_classification",
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.57.6",
  "vocab_size": 30522,
  "id2label": {
    "0": "negative",
    "1": "positive"
  }
}

In [42]:
newmodel = pipeline("text-classification", model='my_saved_model', device='mps')

Device set to use mps


In [43]:
newmodel('This movie is great!')

[{'label': 'positive', 'score': 0.9995844960212708}]

In [44]:
params_after = []
for name,p in model.named_parameters():
    params_after.append(p.detach().cpu().numpy())

In [45]:
for p1, p2 in zip(params_before, params_after):
    print(np.sum(np.abs(p1 - p2)))

13343.374
89.13733
1.7326082
1.1128588
1306.7697
1.6452923
1293.6984
0.003520131
1198.0415
1.0380027
1133.6322
0.85594636
1.6788604
0.86061215
4915.4756
5.693679
4542.0166
0.726169
1.6118168
0.7110889
1287.8215
1.4153147
1282.3904
0.003535627
1098.887
0.8270248
1053.6951
0.7419639
1.4788656
0.72691405
4841.314
5.3331995
4432.135
0.6923027
1.4652106
0.7764585
1250.664
1.5423086
1267.4261
0.0029057255
1108.1886
0.7808902
1083.6248
0.73812
1.5350327
0.7864414
4888.8525
5.4607625
4365.418
0.7149546
1.4561019
0.6682453
1272.7925
1.5559521
1284.7823
0.0032167863
1138.0162
0.75574434
1090.217
0.7303398
1.463409
0.7409211
4781.7837
5.5181274
4107.5312
0.68548006
1.3015227
0.64990795
1192.935
1.505482
1174.1249
0.0021306216
994.4902
0.7989853
996.4249
0.9832226
1.4001925
0.93943727
4425.8115
5.1408405
3515.8538
0.7591144
1.3257899
0.79073024
1090.0786
1.3140922
1105.2164
0.0014150252
944.3025
0.88186777
941.4188
0.97942364
1.3221571
1.1436619
3869.9963
4.7664094
3218.7227
0.9040573
1.3276303
0.